# Experiment No. 07 - StyleGAN Inspired CPU Demonstration

**CPU-friendly Jupyter notebook**

Run cells from top to bottom. These notebooks are designed for an Intel i7 college PC without a dedicated GPU. Some pretrained generative models remain too slow or memory-intensive for CPU; those experiments include lightweight demonstrations and clearly marked optional full-model sections.

## Aim
Demonstrate mapping from latent Z to intermediate W space, style interpolation, and random variation.

**Important:** This is a lightweight StyleGAN-inspired educational GAN, not the full StyleGAN architecture with per-layer style modulation and spatial noise injection.

In [ ]:
%pip install -q tensorflow matplotlib

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Use small random image-like training data so the notebook runs without downloading CelebA.
# Replace with a small face dataset if your instructor requires CelebA specifically.
real_data = tf.random.uniform([256, 16, 16, 3], -1, 1)
ds = tf.data.Dataset.from_tensor_slices(real_data).batch(16)

In [ ]:
mapping = tf.keras.Sequential([
    tf.keras.layers.Input((64,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(64)
])
G = tf.keras.Sequential([
    tf.keras.layers.Input((64,)),
    tf.keras.layers.Dense(8*8*32, activation="relu"),
    tf.keras.layers.Reshape((8,8,32)),
    tf.keras.layers.Conv2DTranspose(16, 4, 2, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(3, 3, padding="same", activation="tanh")
])
D = tf.keras.Sequential([
    tf.keras.layers.Input((16,16,3)),
    tf.keras.layers.Conv2D(16, 4, 2, padding="same", activation="relu"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1)
])
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
go = tf.keras.optimizers.Adam(0.0002)
do = tf.keras.optimizers.Adam(0.0002)

In [ ]:
@tf.function
def train_step(real):
    n = tf.shape(real)[0]
    z = tf.random.normal([n,64])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        w = mapping(z)
        fake = G(w, training=True)
        real_out = D(real, training=True)
        fake_out = D(fake, training=True)
        gl = bce(tf.ones_like(fake_out), fake_out)
        dl = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
    gv = mapping.trainable_variables + G.trainable_variables
    dv = D.trainable_variables
    go.apply_gradients(zip(gt.gradient(gl,gv),gv))
    do.apply_gradients(zip(dt.gradient(dl,dv),dv))
    return gl, dl

for epoch in range(2):
    for batch in ds:
        gl, dl = train_step(batch)
    print("Epoch",epoch+1,"G loss:",float(gl),"D loss:",float(dl))

In [ ]:
z1, z2 = tf.random.normal([1,64]), tf.random.normal([1,64])
w1, w2 = mapping(z1), mapping(z2)
styles = [w1, (w1+w2)/2, w2]
plt.figure(figsize=(9,3))
for i,w in enumerate(styles):
    plt.subplot(1,3,i+1)
    plt.imshow((G(w)[0]+1)/2)
    plt.axis("off")
plt.suptitle("Style interpolation")
plt.show()

## Result
Latent mapping and style interpolation are demonstrated. For genuine CelebA face synthesis and full StyleGAN behavior, use a pretrained StyleGAN implementation on a suitable GPU.